# Validation: COMSOL vs. Analytical Model

Compares the heat flux $Q(N)$ obtained from the COMSOL parametric sweep against
the analytical model (`scripts/analytical_model.py`) for $N = 5$ to $50$ layers.

Replace the `comsol_data` values below with your actual exported CSV
(`data/raw/comsol_sweep_results.csv`) once you have it.

In [ ]:
import sys
sys.path.append('../scripts')

import pandas as pd
import matplotlib.pyplot as plt

from analytical_model import MLIParameters, build_table

In [ ]:
# --- COMSOL results (replace with pd.read_csv('../data/raw/comsol_sweep_results.csv')) ---
comsol_data = {
    'N': [5, 10, 15, 20, 25, 30, 40, 50],
    'Q_comsol_W': [173.36, 86.679, 57.79, 43.339, 34.67, 28.893, 21.67, 17.336],
}
df_comsol = pd.DataFrame(comsol_data)
df_comsol

In [ ]:
# --- Analytical results for the same N values ---
params = MLIParameters()
rows = build_table(df_comsol['N'].tolist(), params)
df_analytical = pd.DataFrame(rows)[['N', 'Q_W']].rename(columns={'Q_W': 'Q_analytical_W'})
df_analytical

In [ ]:
# --- Merge and compute relative error ---
df = df_comsol.merge(df_analytical, on='N')
df['relative_error_pct'] = (
    (df['Q_comsol_W'] - df['Q_analytical_W']).abs() / df['Q_analytical_W'] * 100
)
df

In [ ]:
print(f"Max relative error: {df['relative_error_pct'].max():.4f} %")
print(f"Mean relative error: {df['relative_error_pct'].mean():.4f} %")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(df['N'], df['Q_comsol_W'], 'o-', label='COMSOL')
ax1.plot(df['N'], df['Q_analytical_W'], 's--', label='Analytical')
ax1.set_xlabel('Number of layers, N')
ax1.set_ylabel('Heat flux Q (W)')
ax1.set_title('COMSOL vs. Analytical')
ax1.legend()
ax1.grid(alpha=0.4)

ax2.bar(df['N'].astype(str), df['relative_error_pct'])
ax2.set_xlabel('Number of layers, N')
ax2.set_ylabel('Relative error (%)')
ax2.set_title('Relative error, COMSOL vs. Analytical')
ax2.grid(axis='y', alpha=0.4)

fig.tight_layout()
fig.savefig('../figures/validation_comparison.png', dpi=200)
plt.show()

## Notes

- A near-zero relative error confirms that the COMSOL model correctly implements
  the equivalent conductivity $k_{eff}$ / resistance $R_s$ derived analytically.
- This validates the **numerical implementation only** — it is not an
  independent physical validation of the ideal radiative model itself
  (no comparison against experimental data or a surface-to-surface radiation model).